# Practical Interpolation for Data Preprocessing  
### Lagrange Interpolation + Real-World Use Cases (with Exercises)

**Goal:** learn interpolation as a *data preprocessing tool* to:
- fill missing values (sensor/health data),
- resample signals,
- prepare clean inputs for optimization / ML models.

> **Key message:** Interpolation is not only “math”; it is a practical step before forecasting, anomaly detection, optimization, and control.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred)**2))


## 1. What is interpolation?

Given known data points:

$$
(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n),
$$

interpolation constructs a function $\hat{y}(x)$ such that:

$$
\hat{y}(x_i) = y_i \quad \text{for all } i.
$$

### Why it matters in data preprocessing
Real datasets often contain:
- missing timestamps (sensor dropouts),
- irregular sampling,
- gaps due to network failures or manual recording.

Interpolation helps produce a complete, regular series for:
- feature extraction (rolling stats, derivatives),
- optimization (control, calibration),
- ML training (many models require no missing values).


## 2. Lagrange interpolation (core)

The Lagrange polynomial of degree $n$ is:

$$
P_n(x) = \sum_{i=0}^{n} y_i \, L_i(x),
$$

where each basis polynomial is:

$$
L_i(x) = \prod_{j=0,\,j\neq i}^{n} \frac{x - x_j}{x_i - x_j}.
$$

**Important practical note:**  
High-degree global polynomials can oscillate (Runge phenomenon).  
So in practice, we often do **local Lagrange interpolation** (use a few nearby points).


In [ ]:
def lagrange_poly(x_nodes, y_nodes):
    """Return a function P(x) that evaluates the Lagrange interpolating polynomial
    through (x_nodes, y_nodes). Suitable for small n (local interpolation)."""
    x_nodes = np.asarray(x_nodes, dtype=float)
    y_nodes = np.asarray(y_nodes, dtype=float)
    n = len(x_nodes)

    def P(x):
        x = np.asarray(x, dtype=float)
        total = np.zeros_like(x, dtype=float)
        for i in range(n):
            Li = np.ones_like(x, dtype=float)
            for j in range(n):
                if j == i:
                    continue
                Li *= (x - x_nodes[j])/(x_nodes[i] - x_nodes[j])
            total += y_nodes[i] * Li
        return total

    return P


## 3. Real-world-style dataset: temperature sensor with missing values

Imagine an IoT temperature sensor (cold-chain, hospital lab, server room).
- We have hourly measurements
- Some hours are missing due to connectivity issues

We will:
1. simulate a realistic signal,
2. delete some points (missing data),
3. reconstruct missing values using **local Lagrange interpolation**,
4. show why this matters for downstream analysis.


In [ ]:
# Simulate hourly time points
t = np.arange(0, 48)  # 48 hours

# "True" signal (unknown in reality): daily cycle + small trend + noise
rng = np.random.default_rng(7)
true_temp = 24 + 2.5*np.sin(2*np.pi*t/24) + 0.03*t + rng.normal(0, 0.25, size=len(t))

# Create missingness
missing_idx = np.array([5, 6, 7, 18, 19, 33, 34, 35, 36])  # gaps
observed_mask = np.ones_like(t, dtype=bool)
observed_mask[missing_idx] = False

t_obs = t[observed_mask]
y_obs = true_temp[observed_mask]

plt.figure(figsize=(9,4))
plt.plot(t, true_temp, label="True (hidden)")
plt.scatter(t_obs, y_obs, label="Observed (with missing)", zorder=3)
plt.xlabel("Hour")
plt.ylabel("Temperature (°C)")
plt.title("Sensor series with missing values")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print("Missing hours:", missing_idx.tolist())


## 4. Local Lagrange interpolation for missing timestamps

**Strategy (practical):**  
For each missing time $t_m$, pick the nearest $k$ observed points (e.g., $k=4$ or $k=5$),
build a small-degree Lagrange polynomial, and evaluate it at $t_m$.

This is more stable than fitting one big polynomial to all points.


In [ ]:
def local_lagrange_fill(t_all, t_obs, y_obs, k=4):
    """Fill missing points in t_all using local Lagrange interpolation based on k nearest observed points."""
    t_all = np.asarray(t_all, dtype=float)
    t_obs = np.asarray(t_obs, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float)

    y_filled = np.full_like(t_all, fill_value=np.nan, dtype=float)
    obs_map = {int(tt): yy for tt, yy in zip(t_obs, y_obs)}

    for i, tt in enumerate(t_all.astype(int)):
        if tt in obs_map:
            y_filled[i] = obs_map[tt]
        else:
            d = np.abs(t_obs - tt)
            idx = np.argsort(d)[:k]
            P = lagrange_poly(t_obs[idx], y_obs[idx])
            y_filled[i] = float(P(tt))
    return y_filled

y_lagr = local_lagrange_fill(t, t_obs, y_obs, k=4)

plt.figure(figsize=(9,4))
plt.plot(t, true_temp, label="True (hidden)")
plt.scatter(t_obs, y_obs, label="Observed", zorder=3)
plt.plot(t, y_lagr, '--', label="Filled (local Lagrange, k=4)")
plt.xlabel("Hour")
plt.ylabel("Temperature (°C)")
plt.title("Missing-value imputation by local Lagrange interpolation")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print("RMSE on missing points only:",
      rmse(true_temp[~observed_mask], y_lagr[~observed_mask]))


## 5. Compare with simpler interpolation (baseline)

A simple baseline is **piecewise linear interpolation**.

Why compare?
- In practice, engineers choose the simplest method that works.
- Higher-order interpolation is not always better.

We'll implement linear interpolation with `np.interp`.


In [ ]:
y_linear = np.interp(t, t_obs, y_obs)

plt.figure(figsize=(9,4))
plt.plot(t, true_temp, label="True (hidden)")
plt.plot(t, y_linear, label="Filled (linear)")
plt.plot(t, y_lagr, '--', label="Filled (local Lagrange)")
plt.scatter(t_obs, y_obs, s=18, label="Observed", zorder=3)
plt.xlabel("Hour")
plt.ylabel("Temperature (°C)")
plt.title("Linear vs Local Lagrange for filling missing sensor values")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print("RMSE on missing points (linear): ", rmse(true_temp[~observed_mask], y_linear[~observed_mask]))
print("RMSE on missing points (Lagrange):", rmse(true_temp[~observed_mask], y_lagr[~observed_mask]))


## 6. Why this matters for optimization (practical link)

Many optimization tasks require **clean, complete data**, e.g.:

### Example: choosing a cooling setpoint to minimize cost
Suppose energy cost depends on how much the temperature exceeds a target:

$$
J(s) = \sum_{t} \max(0,\, T(t) - s)^2
$$

where:
- $T(t)$ is measured/cleaned temperature,
- $s$ is a setpoint (decision variable).

If $T(t)$ has missing values, we cannot compute $J(s)$ correctly.

Let's compute the optimal setpoint with:
- the linear-filled series,
- the Lagrange-filled series,
and compare.


In [ ]:
def objective_setpoint(T, s):
    return np.sum(np.maximum(0, T - s)**2)

s_grid = np.linspace(22, 28, 241)

J_lin = np.array([objective_setpoint(y_linear, s) for s in s_grid])
J_lag = np.array([objective_setpoint(y_lagr, s) for s in s_grid])

s_star_lin = s_grid[np.argmin(J_lin)]
s_star_lag = s_grid[np.argmin(J_lag)]

plt.figure(figsize=(8,4))
plt.plot(s_grid, J_lin, label="Objective using linear-filled T(t)")
plt.plot(s_grid, J_lag, label="Objective using Lagrange-filled T(t)")
plt.axvline(s_star_lin, linestyle='--', linewidth=1, label=f"best s (linear)={s_star_lin:.2f}")
plt.axvline(s_star_lag, linestyle='--', linewidth=1, label=f"best s (Lagrange)={s_star_lag:.2f}")
plt.xlabel("Setpoint s (°C)")
plt.ylabel("Cost J(s)")
plt.title("Optimization result depends on preprocessing (interpolation)")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print("Optimal setpoint from linear-filled data:  ", s_star_lin)
print("Optimal setpoint from Lagrange-filled data:", s_star_lag)


## 8. Exercises (with solutions)

### Exercise 1 (missing-value fill)
1. Create a larger missing gap (remove hours 20–26).
2. Fill using local Lagrange with $k=4$ and $k=6$.
3. Compare RMSE on missing points.

✅ Solution below.


In [ ]:
# Exercise 1 — Solution
missing2 = np.arange(20, 27)
mask2 = np.ones_like(t, dtype=bool)
mask2[missing2] = False

t_obs2 = t[mask2]
y_obs2 = true_temp[mask2]

y_lag_k4 = local_lagrange_fill(t, t_obs2, y_obs2, k=4)
y_lag_k6 = local_lagrange_fill(t, t_obs2, y_obs2, k=6)
y_lin2   = np.interp(t, t_obs2, y_obs2)

rm_k4 = rmse(true_temp[~mask2], y_lag_k4[~mask2])
rm_k6 = rmse(true_temp[~mask2], y_lag_k6[~mask2])
rm_lin= rmse(true_temp[~mask2], y_lin2[~mask2])

plt.figure(figsize=(9,4))
plt.plot(t, true_temp, label="True")
plt.scatter(t_obs2, y_obs2, label="Observed", zorder=3)
plt.plot(t, y_lin2, label="Linear fill")
plt.plot(t, y_lag_k4, '--', label="Local Lagrange k=4")
plt.plot(t, y_lag_k6, '--', label="Local Lagrange k=6")
plt.title("Exercise 1: effect of window size k")
plt.xlabel("Hour")
plt.ylabel("Temperature (°C)")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

print("RMSE on missing hours 20..26")
print("Linear:", rm_lin)
print("Lagrange k=4:", rm_k4)
print("Lagrange k=6:", rm_k6)


### Exercise 3 (mini optimization)
Using the filled temperature series, find the setpoint $s$ that minimizes:

$$
J(s)=\sum_t \max(0, T(t)-s)^2.
$$

✅ Solution was implemented in Section 6 (review that cell).


## Takeaways 

- Interpolation builds values between known points: **data → continuous model**.
- In preprocessing, it is used to **fill missing values** and **resample**.
- **Lagrange** is a clear method; in practice use it **locally** (few points).
- Always compare with simple baselines (linear).
- Interpolation affects downstream tasks: features and optimization decisions.

**Class activity:** compare linear vs local Lagrange, then show impact on an optimization result (setpoint).
